# Geospatial Analysis — Grid Intensities & Supply Chains

In [ ]:
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')

In [ ]:
with open(os.path.join(DATA_DIR, 'epa-ghg-emission-factors.json')) as f:
    epa = json.load(f)

# Combine US regions and international grids
us_regions = epa['electricity']['regions']
intl_grids = epa['electricity']['international']

print(f'US grid regions: {len(us_regions)}')
print(f'International grids: {len(intl_grids)}')

## Grid Emission Intensities

In [ ]:
# International grid intensities
intl_rows = []
for code, data in intl_grids.items():
    intl_rows.append({
        'country_code': code,
        'name': data['name'],
        'kgCO2ePerKwh': data['kgCO2ePerKwh']
    })

df_intl = pd.DataFrame(intl_rows).sort_values('kgCO2ePerKwh', ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(df_intl)))
bars = ax.barh(df_intl['name'], df_intl['kgCO2ePerKwh'], color=colors, edgecolor='white')

us_avg = epa['electricity']['us_average']['kgCO2ePerKwh']
ax.axvline(us_avg, color='#e74c3c', linestyle='--', linewidth=2,
           label=f'US Average ({us_avg} kgCO\u2082e/kWh)')

ax.set_xlabel('Grid Intensity (kgCO\u2082e / kWh)')
ax.set_title('International Grid Emission Intensities')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('../figures/grid_intensities.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Cleanest grid: {df_intl.iloc[0]["name"]} ({df_intl.iloc[0]["kgCO2ePerKwh"]} kgCO\u2082e/kWh)')
print(f'Dirtiest grid: {df_intl.iloc[-1]["name"]} ({df_intl.iloc[-1]["kgCO2ePerKwh"]} kgCO\u2082e/kWh)')
print(f'Ratio: {df_intl.iloc[-1]["kgCO2ePerKwh"] / df_intl.iloc[0]["kgCO2ePerKwh"]:.0f}x')

## Supply Chain Distances

In [ ]:
with open(os.path.join(DATA_DIR, 'epa-supply-chain-factors.json')) as f:
    supply_chain = json.load(f)

sc_rows = []
for naics, data in supply_chain['factors'].items():
    sc_rows.append({
        'naics': naics,
        'name': data['name'],
        'kgCO2ePerUsd': data['kgCO2ePerUsd']
    })

df_sc = pd.DataFrame(sc_rows).sort_values('kgCO2ePerUsd', ascending=False)

# Top 20 most carbon-intensive supply chain sectors
top_n = 20
df_top = df_sc.head(top_n)

fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.Reds(np.linspace(0.3, 0.9, top_n))
ax.barh(df_top['name'], df_top['kgCO2ePerUsd'], color=colors, edgecolor='white')
ax.set_xlabel('Supply Chain Intensity (kgCO\u2082e / USD)')
ax.set_title(f'Top {top_n} Most Carbon-Intensive Supply Chain Sectors')
ax.grid(True, alpha=0.3, axis='x')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../figures/supply_chain_intensity.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Total sectors: {len(df_sc)}')
print(f'Mean intensity: {df_sc["kgCO2ePerUsd"].mean():.3f} kgCO\u2082e/USD')
print(f'Median intensity: {df_sc["kgCO2ePerUsd"].median():.3f} kgCO\u2082e/USD')

## Key Observations

- **Grid intensity varies by 50x+** between the cleanest (Sweden, France) and dirtiest (India, Indonesia) grids.
- Manufacturing location choice has a major impact on the electricity-related emissions of a product.
- **Cattle ranching and petroleum products** dominate supply chain intensity rankings.
- Shifting production to low-carbon grids can reduce the energy phase footprint by over 90%.